# 02 · Linear regression: the model ladder

**Status: skeleton, not started.** Follow the guide: [`docs/modeling/02_regression_ladder.md`](../docs/modeling/02_regression_ladder.md).

**Question:** Which groups of predictors explain zone-hour demand, and how much does each add?

**Reads:** `model_sample.parquet` · **Saves:** T06, T07

Conventions (README): same rows for every compared model; never touch the test weeks before 07; save figures and tables under their register ID; end with Findings.

## Setup

In [ ]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

DATA = Path("../data/processed")
FIGURES = Path("../outputs/figures")
TABLES = Path("../outputs/tables")
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
SEED = 42

con = duckdb.connect()


def read_parquet(name):
    """Load a file from data/processed/ with DuckDB (pandas would need pyarrow for Parquet)."""
    return con.execute(f"SELECT * FROM '{(DATA / name).as_posix()}'").df()

In [ ]:
sample = read_parquet("model_sample.parquet")
train = sample[sample["split"] == "train"]
val = sample[sample["split"] == "val"]
print(sample["split"].value_counts())  # expect train 236,350 / val 100,800 / test 100,750 (50 zones)

## 1. Ladder formulas

One definition for every notebook. 03, 04 and 06 must use exactly these.

In [ ]:
TARGET = "y_log"  # change if 03 picks another target (open decision 2)
WEATHER = "temperature_2m + precipitation + snowfall + snow_depth + cloud_cover + shortwave_radiation + is_day"
BLOCKS = {
    "M1": "C(hod)",
    "M2": "C(dow)",
    "M3": "C(month)",
    "M4": "C(zone)",
    "M5": WEATHER,
    "M6": "C(wknd_hod)",  # weekend daily curve; NOT C(hod):weekend (dummy trap, see the guide)
}
LADDER = {"M0": f"{TARGET} ~ 1"} | {
    name: f"{TARGET} ~ " + " + ".join(list(BLOCKS.values())[: i + 1]) for i, name in enumerate(BLOCKS)
}
EXPECTED_PARAMS = {"M0": 1, "M1": 24, "M2": 30, "M3": 41, "M4": 90, "M5": 97, "M6": 120}  # with 50 zones
LADDER

## 2. Fit M0–M6 on train

Check `int(m.df_model) + 1` against `EXPECTED_PARAMS` and that the design matrix has full rank.

In [ ]:
# TODO

## 3. Day-clustered standard errors for M5 and M6

`fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(train["date"])[0]}, use_t=True)`. Add ISO-week clustering as a check if F12 (03) shows correlation across days.

In [ ]:
# TODO

## 4. T06: ladder table

Params, R², adjusted R², RSS per model.

In [ ]:
# TODO

## 5. T07: weather coefficients

M5: coefficient, classic SE, clustered SE, clustered p-value, % effect = 100 × (exp(b) − 1).

In [ ]:
# TODO

## 6. Extensions (optional)

Holiday, temperature², zone × hour (memory: see the guide), lagged demand (only if the horizon allows), rain × hour.

In [ ]:
# TODO

## Findings

_Fill in after running: 3–6 bullets with the numbers the report needs. Name the model, sample and standard-error type for each._

- 